In [ ]:
from dataclasses import asdict

import torch as t
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models

from pathlib import Path

from torchinfo import summary

from aim import Run

from train import train, TrainConfig, TrainJob
from eval import evaluate, plot_loss, plot_metrics

from backbone.resnet20 import ResNet20 

In [ ]:
def mem(label):                                                                            
  alloc = t.cuda.memory_allocated() / 1e9                                            
  res   = t.cuda.memory_reserved()  / 1e9                                            
  peak  = t.cuda.max_memory_allocated() / 1e9                                        
  print(f"{label:20s}  allocated={alloc:.3f} GB  reserved={res:.3f} GB  peak={peak:.3f} GB")                                                                                         
                                                                                              
t.cuda.reset_peak_memory_stats() 

# Setup

### Configs

In [ ]:
train_cfg = TrainConfig(
  epochs=2,
  lr=1e-3,
  weight_decay=1e-4,
  seed=0,
  dataset_split_seed=0,
  device="cuda" if t.cuda.is_available() else "cpu",
)

t.manual_seed(train_cfg.seed)
t.cuda.manual_seed_all(train_cfg.seed)
print(f"device: {train_cfg.device}")

### Load ckpts + models

In [ ]:
ckpt_paths = ["backbone/checkpoints/d1_backbone_seed42_epoch200.pt", "backbone/checkpoints/d1_backbone_seed137_epoch200.pt"]

n_models = 2

assert n_models == len(ckpt_paths)

ckpts = [t.load(ckpt_paths[i], map_location=train_cfg.device, weights_only=True) for i in range(n_models)]

In [ ]:
models = [ResNet20() for _ in range(n_models)]

for i in range(n_models):
  models[i].load_state_dict(ckpts[i]["state_dict"])
  models[i].to(train_cfg.device)

### Load data

In [ ]:
batch_size = 128
image_shape = (1, 3, 32, 32)
num_classes = 100

_CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
_CIFAR100_STD = (0.2675, 0.2565, 0.2761)

train_tf = transforms.Compose([
  transforms.RandomHorizontalFlip(),
  transforms.ToTensor(),
  transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
])
test_tf = transforms.Compose([
  transforms.ToTensor(),
  transforms.Normalize(_CIFAR100_MEAN, _CIFAR100_STD),
])

raw_dataset = datasets.CIFAR100("./data", train=False, download=True)

post_train, post_test = random_split(
  raw_dataset, (0.8, 0.2),
  generator=t.Generator().manual_seed(train_cfg.dataset_split_seed)
)


# save indices
split_path = Path(f"checkpoints/posttrain_split_seed{train_cfg.dataset_split_seed}.pt")          
split_path.parent.mkdir(parents=True, exist_ok=True)                                   
t.save(
  {
    "train_indices": post_train.indices,                                     
    "test_indices":  post_test.indices,
    "split_seed":    train_cfg.dataset_split_seed,
    "source":        "cifar100_test_set",                                          
  },
  split_path,
)

train_dataset = datasets.CIFAR100("./data", train=False, transform=train_tf, download=True)
test_dataset = datasets.CIFAR100("./data", train=False, transform=test_tf, download=True)

post_train = Subset(train_dataset, post_train.indices)
post_test = Subset(test_dataset, post_test.indices)

post_train_loader = DataLoader(post_train, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
post_test_loader = DataLoader(post_test, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

### Setup orchestrator

In [ ]:
from modules.driver import Driver
from modules.bus import Bus, Encoder, Decoder
from modules.orchestrator import Specialist, Orchestrator

In [ ]:
stats = summary(models[0], input_size=image_shape)

stats

In [ ]:
def get_info(m: nn.Module):
  for info in stats.summary_list:
    if info.module is m:
      return info

In [ ]:
early_name = "layer1.2"
late_name = "layer3.0"

# assume homogeny
drivers = [Driver(models[i], early_name, late_name) for i in range(n_models)]

# early <- decoder: want input shape. output shape for encoder
early_shape = get_info(drivers[0].early).input_size
late_shape = get_info(drivers[0].late).output_size

# shape: [B, C, H, W]

early_shape, late_shape

In [ ]:
from einops.layers.torch import Reduce, Rearrange

msg_dim = 64

# TODO(ali): try something else as well
expanders = [Rearrange("b c -> b c 1 1") for _ in range(2)]

# try mean pooling. TODO(ali): try flattening
reducers = [Reduce("b c h w -> b c", "mean") for _ in range(2)]


adapters = [
  {
    "decoder": Decoder(msg_dim, early_shape[1], expanders[i]),
    "encoder": Encoder(late_shape[1], msg_dim, reducers[i])
  }
  for i in range(2)
]

In [ ]:
specialists = [Specialist(drivers[i], **adapters[i]) for i in range(n_models)]
bus = Bus(msg_dim)

# the trained target
orchestrator = Orchestrator(specialists, bus, msg_dim, num_classes)

## optimizer + loss

In [ ]:
optimizer = optim.AdamW(orchestrator.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
criterion = nn.CrossEntropyLoss()

## aim run

Hparams logged for filtering/comparing runs later.

In [ ]:
run = Run(experiment="mvp")
run["hparams"] = {
  **asdict(train_cfg),
  "batch_size": batch_size,
  "image_shape": image_shape,
  "num_classes": num_classes,
  "msg_dim": msg_dim,
  "model": "orchestrator",
  "dataset": "cifar100",
  "backbones": ["ResNet20", "ResNet20"]
}
print(f"aim run hash: {run.hash}")

## train

In [ ]:
job = TrainJob(
  model=orchestrator,
  loader=post_train_loader,
  optimizer=optimizer,
  criterion=criterion,
  config=train_cfg,
  run=run,
)

losses = train(job)

## loss curve

In [ ]:
plot_loss(losses, smooth=50)

## evaluate on test

In [ ]:
results = evaluate(orchestrator, post_test_loader, device=train_cfg.device, criterion=criterion)
print(results)
for k, v in results.items():
  run.track(v, name=f"test_{k}")

## metrics

In [ ]:
plot_metrics(results)